# Division of Methyltransferases into Groups #

Goal of this notebook is to divide entries of methyltransferases downloaded from Uniprot into specific groups.
The keyword that was used to get the data: `(reviewed:true) AND (ec:2.1.1.*) AND (keyword:KW-0949) AND (fragment:false) AND (cc_catalytic_activity:"rhea:*")`

#### Path ####

In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent   # notebooks/ -> main
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUT_DIR = PROJECT_ROOT / "data" / "raw" / "MT_division_output"

DATA_DIR, OUT_DIR

(WindowsPath('C:/Users/ASUS/Documents/moje/labwork/bakalarka/github/methyltransferase/data/raw'),
 WindowsPath('C:/Users/ASUS/Documents/moje/labwork/bakalarka/github/methyltransferase/data/raw/MT_division_output'))

### First block of cells ###
Code in these cells will split the methyltransferases based on the __atom which is methylated__ (e. g. O-MT, N-MT, C-MT)

#### Input: ####
1. `MT_EC_grouped.tsv` file with divided EC numbers into groups (taken from https://enzyme.expasy.org/EC/2.1.1.- and then checked while using https://www.brenda-enzymes.org/index.php)
2. `MT_uniprot.tsv` file with methyltransferase entries

#### Output: ####
`.tsv` group files 

In [2]:
import re
from pathlib import Path
import pandas as pd

EC_REGEX = re.compile(r"\b\d+\.\d+\.\d+\.(?:\d+|-)\b")


def parse_grouped_sectioned(path: Path) -> dict[str, str]:
    """Parse sectioned MT_grouped.tsv (# O_MT blocks). Returns ec -> group."""
    ec_to_group: dict[str, str] = {}
    current_group = None

    for raw in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw.strip()
        if not line:
            continue

        if line.startswith("#"):
            m = re.match(r"#\s*([A-Za-z0-9_-]+)", line)
            current_group = m.group(1) if m else None
            continue

        if line.lower().startswith("ec\t"):
            continue

        if not current_group:
            continue

        ec = line.split("\t", 1)[0].strip()
        if EC_REGEX.fullmatch(ec):
            ec_to_group.setdefault(ec, current_group)

    return ec_to_group


def parse_grouped_table(path: Path) -> dict[str, str]:
    """Parse MT_grouped.tsv as a normal table with columns."""
    df = pd.read_csv(path, sep="\t", dtype=str)

    ec_candidates = [c for c in df.columns if c.strip().lower() in {"ec", "ec_number", "ec number"}]
    if not ec_candidates:
        ec_candidates = [c for c in df.columns if "ec" in c.strip().lower()]
    if not ec_candidates:
        raise RuntimeError(f"Could not find an EC column in {path.name}. Columns: {list(df.columns)}")
    ec_col = ec_candidates[0]

    group_candidates = [c for c in df.columns if c.strip().lower() in {"group", "mt_group", "mt group"}]
    if not group_candidates:
        group_candidates = [c for c in df.columns if "group" in c.strip().lower()]
    if not group_candidates:
        raise RuntimeError(f"Could not find a group column in {path.name}. Columns: {list(df.columns)}")
    group_col = group_candidates[0]

    ec_to_group = {}
    for _, row in df.iterrows():
        ec = str(row.get(ec_col, "")).strip()
        group = str(row.get(group_col, "")).strip()
        if EC_REGEX.fullmatch(ec) and group:
            ec_to_group.setdefault(ec, group)

    return ec_to_group


def load_ec_to_group(path: Path) -> dict[str, str]:
    ec_to_group = parse_grouped_sectioned(path)
    if ec_to_group:
        return ec_to_group
    return parse_grouped_table(path)


def decide_row_group(ec_list: list[str], ec_to_group: dict[str, str]) -> str:
    if not ec_list:
        return "NO_EC"

    groups = []
    unknown = 0
    for ec in ec_list:
        g = ec_to_group.get(ec)
        if g is None:
            unknown += 1
        else:
            groups.append(g)

    if not groups and unknown:
        return "UNKNOWN"
    if groups and unknown:
        return "MIXED"

    gs = set(groups)
    if len(gs) == 1:
        return next(iter(gs))
    return "MULTIPLE"


def sanitize_filename(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", s)


In [4]:
from pathlib import Path

# --- set your paths ---
key_path = DATA_DIR / "MT_EC_grouped.tsv" #Path(r"C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\MT3\input\MT_grouped.tsv")   # <-- change
mt_path = DATA_DIR / "MT_uniprot.tsv" #Path(r"C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\MT3\input\MT3.tsv")     # <-- change

# --- set the EC column name in MT.tsv ---
EC_COL = "EC number"  # <-- change if your MT uses a different column

# --- output options ---
out_dir = OUT_DIR / f"{mt_path.stem}_by_group"
WRITE_EMPTY = False

# ---- run ----
ec_to_group = load_ec_to_group(key_path)
print("Loaded EC->group mappings:", len(ec_to_group))

df = pd.read_csv(mt_path, sep="\t", dtype=str)
print("Loaded MT rows:", len(df))
print("MT columns:", list(df.columns))

if EC_COL not in df.columns:
    raise RuntimeError(f"Column '{EC_COL}' not found. Available: {list(df.columns)}")

df["_ec_list"] = df[EC_COL].fillna("").astype(str).apply(lambda s: EC_REGEX.findall(s))
df["MT_group"] = df["_ec_list"].apply(lambda ecs: decide_row_group(ecs, ec_to_group))

out_dir.mkdir(parents=True, exist_ok=True)

key_groups = sorted(set(ec_to_group.values()))
special = ["MULTIPLE", "MIXED", "UNKNOWN", "NO_EC"]
all_groups = key_groups + [g for g in special if g not in key_groups]

written = 0
for g in all_groups:
    sub = df[df["MT_group"] == g].drop(columns=["_ec_list"], errors="ignore")
    if len(sub) == 0 and not WRITE_EMPTY:
        continue
    out_file = out_dir / f"{mt_path.stem}_{sanitize_filename(g)}.tsv"
    sub.to_csv(out_file, sep="\t", index=False)
    written += 1

summary = (
    df["MT_group"]
    .value_counts(dropna=False)
    .rename_axis("MT_group")
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)
summary_file = out_dir / f"{mt_path.stem}_group_counts.tsv"
summary.to_csv(summary_file, sep="\t", index=False)

display(summary)

print(f"Wrote {written} group files to: {out_dir}")
print(f"Wrote summary: {summary_file}")


Loaded EC->group mappings: 399
Loaded MT rows: 11315
MT columns: ['Entry', 'Entry Name', 'Protein names', 'Gene Names', 'Organism', 'Organism (ID)', 'Length', 'Sequence', '3D', 'Protein families', 'Motif', 'Repeat', 'Domain [CC]', 'Domain [FT]', 'Sequence similarities', 'EC number', 'Rhea ID', 'Catalytic activity', 'Function [CC]', 'DOI ID']


,MT_group,rows
0,N_MT,5941
1,C_MT,1922
2,O_MT,1906
3,NO_EC,893
4,MIXED,293
5,MULTIPLE,272
6,S_MT,49
7,UNKNOWN,30
8,OTHER,9


Wrote 9 group files to: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\MT_uniprot_by_group
Wrote summary: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\MT_uniprot_by_group\MT_uniprot_group_counts.tsv


### Second block of cells ###
Code in these cells will split the methyltransferases based on their specific __EC number__.

#### Input: ####
`MT_uniprot.tsv` file with methyltransferase entries

#### Output: ####
`.tsv` specific EC number files 

In [6]:
import re
from pathlib import Path
import pandas as pd

EC_REGEX = re.compile(r"\b\d+\.\d+\.\d+\.(?:\d+|-)\b")

def guess_sep(path: Path) -> str:
    if path.suffix.lower() in {".tsv", ".tab"}:
        return "\t"
    if path.suffix.lower() == ".csv":
        return ","
    return "\t"

def extract_ec_list(s: str) -> list[str]:
    """Return a sorted unique list of EC numbers found in a string."""
    if not isinstance(s, str):
        return []
    ecs = EC_REGEX.findall(s)
    return sorted(set(ecs))

def sanitize_filename(s: str) -> str:
    """Make a safe filename part from an EC key."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s))

def split_uniprot_by_ec(
    input_file,
    sep=None,
    ec_col="EC number",
    cat_col="Catalytic activity",
    mode="explode",                 # "first" | "explode" | "joined"
    out_dir=None
):
    """
    Split a UniProt export table into separate files based on EC number.
    Returns (summary_df, out_dir_path).
    """
    in_path = Path(input_file)
    sep = sep if sep is not None else guess_sep(in_path)

    df = pd.read_csv(in_path, sep=sep, dtype=str)

    # Choose source column for EC extraction
    if ec_col in df.columns:
        ec_source = df[ec_col].fillna("").astype(str)
        source_used = ec_col
    elif cat_col in df.columns:
        ec_source = df[cat_col].fillna("").astype(str)
        source_used = cat_col
    else:
        ec_source = pd.Series([""] * len(df))
        source_used = None

    df["EC_list"] = ec_source.map(extract_ec_list)

    # Group depending on mode
    if mode == "first":
        df_work = df.copy()
        df_work["EC_key"] = df_work["EC_list"].map(lambda xs: xs[0] if xs else "NO_EC")
        groups = df_work.groupby("EC_key", dropna=False)
    elif mode == "joined":
        df_work = df.copy()
        df_work["EC_key"] = df_work["EC_list"].map(lambda xs: "|".join(xs) if xs else "NO_EC")
        groups = df_work.groupby("EC_key", dropna=False)
    elif mode == "explode":
        df_work = df.copy()
        df_work["EC_key"] = df_work["EC_list"]
        df_work = df_work.explode("EC_key")
        df_work["EC_key"] = df_work["EC_key"].fillna("NO_EC")
        groups = df_work.groupby("EC_key", dropna=False)
    else:
        raise ValueError("mode must be one of: 'first', 'explode', 'joined'")

    out_dir = Path(out_dir) if out_dir else in_path.parent / f"{in_path.stem}_ec_split"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Write one file per EC group
    summary_rows = []
    out_suffix = in_path.suffix if in_path.suffix else ".tsv"

    for ec_key, g in groups:
        ec_key_str = str(ec_key)
        safe = sanitize_filename(ec_key_str)
        out_file = out_dir / f"{in_path.stem}_EC_{safe}{out_suffix}"
        g.drop(columns=["EC_list"], errors="ignore").to_csv(out_file, sep=sep, index=False)
        summary_rows.append({"EC_key": ec_key_str, "rows": len(g), "file": out_file.name})

    summary = pd.DataFrame(summary_rows).sort_values(["rows", "EC_key"], ascending=[False, True])
    summary_file = out_dir / f"{in_path.stem}_EC_summary.tsv"
    summary.to_csv(summary_file, sep="\t", index=False)

    # Print a small run report (still notebook-friendly)
    print(f"Input file: {in_path}")
    print(f"Separator: {repr(sep)}")
    if source_used:
        print(f"EC source column: {source_used}")
    else:
        print(f"EC source column: NONE (no '{ec_col}' or '{cat_col}' found)")

    print(f"Input rows: {len(df)}")
    if mode == "explode":
        print("Exploded rows (proteins with multiple EC counted multiple times):", int(summary["rows"].sum()))
    print(f"Output directory: {out_dir}")
    print(f"Summary written: {summary_file}")
    print("Top 10 EC groups:")
    display(summary.head(10))

    return summary, out_dir


In [8]:
from pathlib import Path

input_file = DATA_DIR / "MT_uniprot.tsv"  # <-- change

summary, out_dir = split_uniprot_by_ec(
    input_file=input_file,
    sep=None,                      # None = auto (tsv/csv guess)
    ec_col="EC number",
    cat_col="Catalytic activity",
    mode="explode",                # "explode" (recommended), "first", or "joined"
    out_dir=OUT_DIR / "MT_division_by_EC"      # None = <stem>_ec_split next to input
)

summary.head(20)


Input file: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_uniprot.tsv
Separator: '\t'
EC source column: EC number
Input rows: 11315
Exploded rows (proteins with multiple EC counted multiple times): 12923
Output directory: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\MT_division_by_EC
Summary written: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\MT_division_by_EC\MT_uniprot_EC_summary.tsv
Top 10 EC groups:


,EC_key,rows,file
325,NO_EC,893,MT_uniprot_EC_NO_EC.tsv
86,2.1.1.199,839,MT_uniprot_EC_2.1.1.199.tsv
118,2.1.1.228,705,MT_uniprot_EC_2.1.1.228.tsv
80,2.1.1.192,637,MT_uniprot_EC_2.1.1.192.tsv
70,2.1.1.182,624,MT_uniprot_EC_2.1.1.182.tsv
65,2.1.1.177,618,MT_uniprot_EC_2.1.1.177.tsv
207,2.1.1.33,557,MT_uniprot_EC_2.1.1.33.tsv
53,2.1.1.166,407,MT_uniprot_EC_2.1.1.166.tsv
51,2.1.1.163,386,MT_uniprot_EC_2.1.1.163.tsv
58,2.1.1.170,368,MT_uniprot_EC_2.1.1.170.tsv


,EC_key,rows,file
325,NO_EC,893,MT_uniprot_EC_NO_EC.tsv
86,2.1.1.199,839,MT_uniprot_EC_2.1.1.199.tsv
118,2.1.1.228,705,MT_uniprot_EC_2.1.1.228.tsv
80,2.1.1.192,637,MT_uniprot_EC_2.1.1.192.tsv
70,2.1.1.182,624,MT_uniprot_EC_2.1.1.182.tsv
65,2.1.1.177,618,MT_uniprot_EC_2.1.1.177.tsv
207,2.1.1.33,557,MT_uniprot_EC_2.1.1.33.tsv
53,2.1.1.166,407,MT_uniprot_EC_2.1.1.166.tsv
51,2.1.1.163,386,MT_uniprot_EC_2.1.1.163.tsv
58,2.1.1.170,368,MT_uniprot_EC_2.1.1.170.tsv


### Third block of cells ###
Code in these cells will count the number of __entries under each EC number__. 

#### Input: ####
1. `MT_EC_grouped.tsv` file with split methyltransferase entries (e. g. O-MT, N-MT, S-MT)
2. `MT_uniprot_EC_summary.tsv` file with count of each EC number entry present in the dataset

#### Output: ####
`.tsv` summary file 

In [12]:
import re
from pathlib import Path
import pandas as pd

EC_REGEX = re.compile(r"\b\d+\.\d+\.\d+\.(?:\d+|-)\b")


def parse_mt_grouped(path: Path) -> dict[str, str]:
    """
    Parse MT_grouped.tsv with sections like:
      # O_MT (123)
      ec    name    status
      2.1.1.1 ...

    Returns: ec -> group (e.g. O_MT, N_MT, ...)
    """
    ec_to_group: dict[str, str] = {}
    current_group = None

    for raw in path.read_text(encoding="utf-8", errors="replace").splitlines():
        line = raw.strip()
        if not line:
            continue

        if line.startswith("#"):
            m = re.match(r"#\s*([A-Za-z0-9_]+)\b", line)  # "# O_MT (..)" -> "O_MT"
            current_group = m.group(1) if m else None
            continue

        if line.lower().startswith("ec\t"):
            continue

        if current_group is None:
            continue

        parts = line.split("\t")
        if not parts:
            continue
        ec = parts[0].strip()
        if EC_REGEX.fullmatch(ec):
            ec_to_group.setdefault(ec, current_group)

    return ec_to_group


def choose_group_for_ecs(ecs: list[str], ec_to_group: dict[str, str]) -> str:
    """
    Decide group for a list of ECs found in one summary row.
      - all map to same group -> that group
      - different groups -> MULTIPLE
      - none map -> UNKNOWN
      - mix of mapped + unmapped -> MIXED
      - no ECs found -> NO_EC
    """
    if not ecs:
        return "NO_EC"

    groups = []
    unknown = 0
    for ec in ecs:
        g = ec_to_group.get(ec)
        if g is None:
            unknown += 1
        else:
            groups.append(g)

    if not groups and unknown:
        return "UNKNOWN"
    if groups and unknown:
        return "MIXED"

    gs = set(groups)
    if len(gs) == 1:
        return next(iter(gs))
    return "MULTIPLE"


In [15]:
# --- INPUT FILES (edit paths) ---
mt_grouped_path = DATA_DIR/ "MT_EC_grouped.tsv"
ec_summary_path  = DATA_DIR/ "MT_division_output" / "MT_division_by_EC" / "MT_uniprot_EC_summary.tsv"

# --- COLUMN NAMES ---
EC_COL = "EC_key"   # change if your column is called "EC", etc.
COUNT_COL = "rows"  # change if your count column has a different name; or set to None

# --- OUTPUT FILES (optional: leave None to auto-name next to input) ---
out_summary = OUT_DIR / "EC_summary"  
out_totals  = OUT_DIR / "EC_summary" # e.g. Path(r"C:\path\to\MT2_group_totals.tsv")

print("MT_grouped:", mt_grouped_path)
print("EC_summary:", ec_summary_path)


MT_grouped: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_EC_grouped.tsv
EC_summary: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\MT_division_by_EC\MT_uniprot_EC_summary.tsv


In [16]:
# 1) Load EC -> group mapping
ec_to_group = parse_mt_grouped(mt_grouped_path)
if not ec_to_group:
    raise RuntimeError(
        "Parsed 0 EC->group mappings from MT_grouped.tsv. "
        "Check that it contains '# O_MT (...)' blocks and EC numbers in the first column."
    )

print("Loaded EC->group mappings:", len(ec_to_group))

# 2) Load summary
df = pd.read_csv(ec_summary_path, sep="\t", dtype=str)

if EC_COL not in df.columns:
    raise RuntimeError(f"Column '{EC_COL}' not found. Available columns: {list(df.columns)}")

# 3) Extract ECs from the EC column and assign groups
df["_ecs"] = df[EC_COL].fillna("").astype(str).apply(lambda s: EC_REGEX.findall(s))
df["MT_group"] = df["_ecs"].apply(lambda ecs: choose_group_for_ecs(ecs, ec_to_group))

# 4) Write annotated summary
out_summary = Path(out_summary) if out_summary else ec_summary_path.with_name(f"{ec_summary_path.stem}_with_groups.tsv")
df.drop(columns=["_ecs"], errors="ignore").to_csv(out_summary, sep="\t", index=False)

print("Wrote annotated summary:", out_summary)

# Preview
df.head()


Loaded EC->group mappings: 399
Wrote annotated summary: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\EC_summary


,EC_key,rows,file,_ecs,MT_group
0,NO_EC,893,MT_uniprot_EC_NO_EC.tsv,[],NO_EC
1,2.1.1.199,839,MT_uniprot_EC_2.1.1.199.tsv,[2.1.1.199],N_MT
2,2.1.1.228,705,MT_uniprot_EC_2.1.1.228.tsv,[2.1.1.228],N_MT
3,2.1.1.192,637,MT_uniprot_EC_2.1.1.192.tsv,[2.1.1.192],C_MT
4,2.1.1.182,624,MT_uniprot_EC_2.1.1.182.tsv,[2.1.1.182],N_MT


In [17]:
df_for_totals = df.copy()

if COUNT_COL and COUNT_COL in df_for_totals.columns:
    df_for_totals["_count_num"] = pd.to_numeric(df_for_totals[COUNT_COL], errors="coerce").fillna(0).astype(int)

    totals = (
        df_for_totals.groupby("MT_group", dropna=False)
        .agg(
            total_rows_in_summary=("MT_group", "size"),
            total_count=("_count_num", "sum"),
            distinct_ec_keys=(EC_COL, pd.Series.nunique),
        )
        .reset_index()
        .sort_values(["total_count", "total_rows_in_summary"], ascending=False)
    )
else:
    totals = (
        df_for_totals.groupby("MT_group", dropna=False)
        .agg(
            total_rows_in_summary=("MT_group", "size"),
            distinct_ec_keys=(EC_COL, pd.Series.nunique),
        )
        .reset_index()
        .sort_values(["total_rows_in_summary"], ascending=False)
    )

out_totals = Path(out_totals) if out_totals else ec_summary_path.with_name(f"{ec_summary_path.stem}_group_totals.tsv")
totals.to_csv(out_totals, sep="\t", index=False)

print("Wrote totals:", out_totals)
display(totals)


Wrote totals: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\EC_summary


,MT_group,total_rows_in_summary,total_count,distinct_ec_keys
2,N_MT,120,6308,120
4,O_MT,120,2535,120
0,C_MT,50,2338,50
1,NO_EC,1,893,1
6,UNKNOWN,25,786,25
5,S_MT,8,54,8
3,OTHER,2,9,2


### Fourth block of cells ###
Code in these cells will count the number of __unique base entry names__ (same type of enzymes have different entry names depending on the organism from which they were isolated, e. g. POLN_SINDV, POLN_SFV, POLN_ONNVG; I wanted to find out how many different kinds of enzymes are present in the dataset).  

#### Input: ####
`MT_uniprot_by_group/*.tsv` files with split methyltransferase entries (e. g. O-MT, N-MT, S-MT)

#### Output: ####
`unique_entry_base_summary.tsv` file 

In [18]:
from pathlib import Path
from collections import Counter
import pandas as pd

def base_name(entry: str) -> str:
    """Return the base part before the first underscore."""
    entry = str(entry).strip()
    if not entry:
        return ""
    return entry.split("_", 1)[0]


In [20]:
# Option A: list files explicitly

#files = [
#    Path(r"C:\path\to\O_MT.tsv"),
#    Path(r"C:\path\to\N_MT.tsv"),
#    # Path(r"..."),
#]


# Option B: glob a folder (uncomment and edit)
files = sorted((OUT_DIR / "MT_uniprot_by_group" ).glob("*.tsv"))
files = [p for p in files if not p.name.endswith("_group_counts.tsv")]

COL = "Entry Name"   # column containing UniProt entry names
TOP = 15             # top N base names to show; set 0 to skip

print("Number of files:", len(files))
files[:5]


Number of files: 9


[WindowsPath('C:/Users/ASUS/Documents/moje/labwork/bakalarka/datasety/data-science-template-main/data/raw/MT_division_output/MT_uniprot_by_group/MT_uniprot_C_MT.tsv'),
 WindowsPath('C:/Users/ASUS/Documents/moje/labwork/bakalarka/datasety/data-science-template-main/data/raw/MT_division_output/MT_uniprot_by_group/MT_uniprot_MIXED.tsv'),
 WindowsPath('C:/Users/ASUS/Documents/moje/labwork/bakalarka/datasety/data-science-template-main/data/raw/MT_division_output/MT_uniprot_by_group/MT_uniprot_MULTIPLE.tsv'),
 WindowsPath('C:/Users/ASUS/Documents/moje/labwork/bakalarka/datasety/data-science-template-main/data/raw/MT_division_output/MT_uniprot_by_group/MT_uniprot_N_MT.tsv'),
 WindowsPath('C:/Users/ASUS/Documents/moje/labwork/bakalarka/datasety/data-science-template-main/data/raw/MT_division_output/MT_uniprot_by_group/MT_uniprot_NO_EC.tsv')]

In [21]:
overall_full = set()
overall_base = set()
overall_base_counts = Counter()

per_file_rows = []  # for a nice summary table

for path in files:
    path = Path(path)
    df = pd.read_csv(path, sep="\t", dtype=str)

    if COL not in df.columns:
        raise ValueError(f"Column '{COL}' not found in {path.name}. Columns: {list(df.columns)}")

    entries = df[COL].fillna("").astype(str).str.strip()
    entries = entries[entries != ""]  # drop empty

    full_set = set(entries.tolist())

    bases = []
    for x in entries.tolist():
        b = base_name(x)
        if b:
            bases.append(b)

    base_set = set(bases)
    base_counts = Counter(bases)

    overall_full |= full_set
    overall_base |= base_set
    overall_base_counts.update(base_counts)

    per_file_rows.append({
        "file": path.name,
        "rows_nonempty": len(entries),
        "unique_full_entry_names": len(full_set),
        "unique_base_names": len(base_set),
    })

    print(f"\n== {path.name} ==")
    print(f"Rows (non-empty '{COL}'): {len(entries)}")
    print(f"Unique full entry names:       {len(full_set)}")
    print(f"Unique base names (before _):  {len(base_set)}")

    if TOP and base_counts:
        print(f"Top {TOP} base names:")
        for name, cnt in base_counts.most_common(TOP):
            print(f"  {name}\t{cnt}")

print("\n== OVERALL (across all files) ==")
print(f"Unique full entry names:      {len(overall_full)}")
print(f"Unique base names (before _): {len(overall_base)}")

if TOP and overall_base_counts:
    print(f"Top {TOP} base names overall:")
    for name, cnt in overall_base_counts.most_common(TOP):
        print(f"  {name}\t{cnt}")

summary_df = pd.DataFrame(per_file_rows).sort_values("rows_nonempty", ascending=False)
display(summary_df)



== MT_uniprot_C_MT.tsv ==
Rows (non-empty 'Entry Name'): 1922
Unique full entry names:       1922
Unique base names (before _):  151
Top 15 base names:
  RLMN	630
  TRMA	183
  CBID	182
  RLMD	161
  MENG	126
  RLMC	96
  RSMF	87
  RSMB	84
  RLMI	81
  CBIT	27
  SUMT	13
  ERG6	11
  CFR	11
  DNMT1	7
  TRDMT	6

== MT_uniprot_MIXED.tsv ==
Rows (non-empty 'Entry Name'): 293
Unique full entry names:       293
Unique base names (before _):  27
Top 15 base names:
  CYSG	116
  POLG	59
  VP3	32
  TYW4	14
  MCEL	13
  CYSG1	12
  CYSG2	12
  MCE	6
  LMBD2	3
  BIOHC	3
  L	3
  VP1	3
  EGT1	2
  TYW23	2
  WBDD	1

== MT_uniprot_MULTIPLE.tsv ==
Rows (non-empty 'Entry Name'): 272
Unique full entry names:       272
Unique base names (before _):  2
Top 15 base names:
  UBIE	268
  INMT	4

== MT_uniprot_N_MT.tsv ==
Rows (non-empty 'Entry Name'): 5941
Unique full entry names:       5941
Unique base names (before _):  353
Top 15 base names:
  RSMH	831
  TRMD	642
  RSMA	624
  RLMH	610
  TRMB	550
  RSMG	363
  RSMJ	1

,file,rows_nonempty,unique_full_entry_names,unique_base_names
3,MT_uniprot_N_MT.tsv,5941,5941,353
0,MT_uniprot_C_MT.tsv,1922,1922,151
5,MT_uniprot_O_MT.tsv,1906,1906,184
4,MT_uniprot_NO_EC.tsv,893,893,170
1,MT_uniprot_MIXED.tsv,293,293,27
2,MT_uniprot_MULTIPLE.tsv,272,272,2
7,MT_uniprot_S_MT.tsv,49,49,19
8,MT_uniprot_UNKNOWN.tsv,30,30,6
6,MT_uniprot_OTHER.tsv,9,9,3


In [23]:
out_summary = OUT_DIR / "unique_entry_base_summary.tsv"  # change path if needed
summary_df.to_csv(out_summary, sep="\t", index=False)
print("Wrote:", out_summary.resolve())


Wrote: C:\Users\ASUS\Documents\moje\labwork\bakalarka\datasety\data-science-template-main\data\raw\MT_division_output\unique_entry_base_summary.tsv


### Fifth block of cells ###
Counting how many different Rhea numbers (= how many unique reactions) there are in each category of methyltransferases. 

#### Input ####
`MT_uniprot_by_group/*.tsv` files with split methyltransferase entries (e. g. O-MT, N-MT, S-MT)

#### Output ####


In [1]:
from pathlib import Path
import re
import pandas as pd

# Matches Rhea reaction identifiers like RHEA:12345
RHEA_REGEX = re.compile(r"\bRHEA:\d+\b", flags=re.IGNORECASE)

def extract_rhea_list(x) -> list[str]:
    """
    Extract sorted unique Rhea IDs from a cell.
    Handles:
      - 'RHEA:12345'
      - 'RHEA:12345; RHEA:67890'
      - embedded in longer text (Catalytic activity)
      - NaN / None
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    s = str(x)
    hits = RHEA_REGEX.findall(s)
    # Normalize to uppercase
    hits = [h.upper() for h in hits]
    return sorted(set(hits))

def pick_rhea_source_column(df: pd.DataFrame, preferred: str | None = None) -> str | None:
    """
    Pick a column to extract Rhea IDs from.
    - If preferred is given and exists, use it.
    - Else try common Rhea column names.
    - Else fall back to a catalytic-activity-like column if present.
    """
    if preferred and preferred in df.columns:
        return preferred

    # Common explicit Rhea columns
    candidates = []
    for c in df.columns:
        cl = c.strip().lower()
        if cl in {"rhea", "rhea id", "rhea-id", "rhea_ids", "rhea ids"}:
            candidates.append(c)
    if candidates:
        return candidates[0]

    # Fall back to text columns where Rhea may be embedded
    for c in df.columns:
        cl = c.strip().lower()
        if "catalytic" in cl or "reaction" in cl:
            return c

    return None


In [4]:
GROUP_DIR = DATA_DIR / "MT_division_output" / "MT_uniprot_by_group"  # <-- change
files = sorted(GROUP_DIR.glob("*.tsv"))

# Exclude summary files if present
files = [p for p in files if not p.name.endswith("_group_counts.tsv")]

print("Found files:", len(files))
for p in files[:10]:
    print(" -", p.name)


Found files: 9
 - MT_uniprot_C_MT.tsv
 - MT_uniprot_MIXED.tsv
 - MT_uniprot_MULTIPLE.tsv
 - MT_uniprot_N_MT.tsv
 - MT_uniprot_NO_EC.tsv
 - MT_uniprot_O_MT.tsv
 - MT_uniprot_OTHER.tsv
 - MT_uniprot_S_MT.tsv
 - MT_uniprot_UNKNOWN.tsv


In [6]:
# How to get category label from filename:
# This assumes filenames like "MT_O_MT.tsv" or "something_S_MT.tsv".
def category_from_filename(path: Path) -> str:
    stem = path.stem
    # category is the last underscore-separated token(s) in many pipelines, e.g. MT3_O_MT -> O_MT
    # If your filenames differ, adjust this function.
    parts = stem.split("_")
    if len(parts) >= 2 and parts[-2] in {"O", "N", "C", "S"} and parts[-1] == "MT":
        return parts[-2] + "_" + parts[-1]   # O_MT
    # Otherwise try to find known group substrings
    for g in ["O_MT", "N_MT", "C_MT", "S_MT", "UNCLEAR", "OTHER", "MULTIPLE", "MIXED", "UNKNOWN", "NO_EC"]:
        if g in stem:
            return g
    return stem  # fallback

# If you KNOW the exact column name containing Rhea IDs, set it here; else leave None.
RHEA_COL = None  # e.g. "Rhea ID" or "Rhea"


In [11]:
results = []
overall_unique_rhea = set()
overall_multi_rhea_rows = 0
overall_rows = 0

for path in files:
    df = pd.read_csv(path, sep="\t", dtype=str)
    category = category_from_filename(path)

    # choose column to parse
    col = pick_rhea_source_column(df, preferred=RHEA_COL)
    if col is None:
        # No Rhea-like column found; treat as zero Rhea
        df["_rhea_list"] = [[] for _ in range(len(df))]
        col_used = "(none found)"
    else:
        df["_rhea_list"] = df[col].apply(extract_rhea_list)
        col_used = col

    # per-row counts
    rhea_counts = df["_rhea_list"].apply(len)

    n_rows = len(df)
    n_with_rhea = int((rhea_counts >= 1).sum())
    n_multi_rhea = int((rhea_counts >= 2).sum())

    # unique Rhea in category
    cat_unique = set()
    for lst in df["_rhea_list"]:
        cat_unique.update(lst)

    overall_unique_rhea.update(cat_unique)
    overall_multi_rhea_rows += n_multi_rhea
    overall_rows += n_rows

    results.append({
        "category": category,
        "file": path.name,
        "rhea_source_column": col_used,
        "rows_total": n_rows,
       #"rows_with_>=1_rhea": n_with_rhea,
        "rows_with_=1_rhea": n_rows-n_multi_rhea,
        "rows_with_>=2_rhea": n_multi_rhea,
        "unique_rhea_in_category": len(cat_unique),
    })

summary_df = pd.DataFrame(results).sort_values(["category", "file"]).reset_index(drop=True)

display(summary_df)

print("\nOVERALL")
print("Total rows across files:", overall_rows)
print("Total unique Rhea IDs across all categories/files:", len(overall_unique_rhea))
print("Total rows with multiple Rhea IDs (>=2):", overall_multi_rhea_rows)


,category,file,rhea_source_column,rows_total,rows_with_=1_rhea,rows_with_>=2_rhea,unique_rhea_in_category
0,C_MT,MT_uniprot_C_MT.tsv,Rhea ID,1922,1065,857,77
1,MIXED,MT_uniprot_MIXED.tsv,Rhea ID,293,3,290,32
2,MULTIPLE,MT_uniprot_MULTIPLE.tsv,Rhea ID,272,0,272,6
3,NO_EC,MT_uniprot_NO_EC.tsv,Rhea ID,893,781,112,169
4,N_MT,MT_uniprot_N_MT.tsv,Rhea ID,5941,5533,408,192
5,OTHER,MT_uniprot_OTHER.tsv,Rhea ID,9,1,8,4
6,O_MT,MT_uniprot_O_MT.tsv,Rhea ID,1906,1523,383,203
7,S_MT,MT_uniprot_S_MT.tsv,Rhea ID,49,48,1,8
8,UNKNOWN,MT_uniprot_UNKNOWN.tsv,Rhea ID,30,3,27,22



OVERALL
Total rows across files: 11315
Total unique Rhea IDs across all categories/files: 669
Total rows with multiple Rhea IDs (>=2): 2358
